In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
import random

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target
n_features = X.shape[1]

# Evaluation function
def evaluate(state):
    selected = [i for i in range(n_features) if state[i] == 1]
    if len(selected) == 0:
        return 0
    X_subset = X[:, selected]
    model = LogisticRegression(max_iter=3000)
    score = cross_val_score(model, X_subset, y, cv=5).mean()
    return score

# Generate neighbors
def get_neighbors(state):
    neighbors = []
    for i in range(n_features):
        neighbor = state.copy()
        neighbor[i] = 1 - neighbor[i]  # flip bit
        neighbors.append(neighbor)
    return neighbors

# Hill Climbing
def hill_climb():
    current = np.random.randint(0, 2, n_features)
    current_score = evaluate(current)

    while True:
        neighbors = get_neighbors(current)
        scores = [evaluate(n) for n in neighbors]

        best_index = np.argmax(scores)
        best_neighbor = neighbors[best_index]
        best_score = scores[best_index]

        if best_score <= current_score:
            break

        current, current_score = best_neighbor, best_score

    return current, current_score

# Run multiple times
for i in range(3):
    solution, score = hill_climb()
    print(f"Run {i+1}: Accuracy = {score:.4f}, Features Selected = {sum(solution)}")

Run 1: Accuracy = 0.9561, Features Selected = 12
Run 2: Accuracy = 0.9578, Features Selected = 14
Run 3: Accuracy = 0.9596, Features Selected = 16


In [2]:
import numpy as np
import random
import math

# Generate random city coordinates
n_cities = 10
cities = np.random.rand(n_cities, 2) * 100

# Distance function
def total_distance(route):
    dist = 0
    for i in range(len(route)):
        a = cities[route[i]]
        b = cities[route[(i+1) % len(route)]]
        dist += np.linalg.norm(a - b)
    return dist

# Neighbor by swapping two cities
def neighbor(route):
    new_route = route.copy()
    i, j = random.sample(range(len(route)), 2)
    new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route

# Simulated Annealing
def simulated_annealing():
    T = 1000
    cooling = 0.995
    current = list(range(n_cities))
    random.shuffle(current)
    best = current.copy()

    while T > 1:
        new = neighbor(current)
        delta = total_distance(new) - total_distance(current)

        if delta < 0 or random.random() < math.exp(-delta / T):
            current = new

        if total_distance(current) < total_distance(best):
            best = current

        T *= cooling

    return best, total_distance(best)

route, dist = simulated_annealing()
print("Best Route:", route)
print("Total Distance:", dist)

Best Route: [6, 7, 3, 5, 4, 0, 9, 2, 8, 1]
Total Distance: 302.35917011485833


In [3]:
import numpy as np
import random

# Map settings
START = np.array([0, 0])
END = np.array([100, 100])
NUM_WAYPOINTS = 5
POP_SIZE = 30
GENERATIONS = 100

# Obstacles (circles)
obstacles = [(50, 50, 15), (70, 30, 10)]

# Create random chromosome
def create_path():
    return np.random.rand(NUM_WAYPOINTS, 2) * 100

# Fitness function
def fitness(path):
    points = [START] + list(path) + [END]
    dist = 0
    penalty = 0

    for i in range(len(points) - 1):
        dist += np.linalg.norm(points[i] - points[i+1])

    # obstacle penalty
    for p in path:
        for ox, oy, r in obstacles:
            if np.linalg.norm(p - np.array([ox, oy])) < r:
                penalty += 1000

    return dist + penalty

# Tournament selection
def select(pop):
    a, b = random.sample(pop, 2)
    return a if fitness(a) < fitness(b) else b

# Crossover
def crossover(p1, p2):
    point = random.randint(1, NUM_WAYPOINTS - 1)
    child = np.vstack((p1[:point], p2[point:]))
    return child

# Mutation
def mutate(path):
    idx = random.randint(0, NUM_WAYPOINTS - 1)
    path[idx] += np.random.randn(2) * 5
    path[idx] = np.clip(path[idx], 0, 100)
    return path

# GA main loop
population = [create_path() for _ in range(POP_SIZE)]

for _ in range(GENERATIONS):
    new_pop = []
    for _ in range(POP_SIZE):
        p1 = select(population)
        p2 = select(population)
        child = crossover(p1, p2)
        child = mutate(child)
        new_pop.append(child)
    population = new_pop

best = min(population, key=fitness)
print("Best Path Fitness:", fitness(best))
print("Waypoints:\n", best)

Best Path Fitness: 142.31959982613216
Waypoints:
 [[17.63195351 25.45564786]
 [57.79531164 65.45924249]
 [70.98195254 76.54214285]
 [87.41352008 88.45310263]
 [94.88418052 96.29365103]]
